# Scratch-coder Stage A aggregate review

This notebook never displays raw response rows, source IDs, projects, or disagreement cases.

## 1. Frozen input

In [ ]:
from pathlib import Path
import json, pandas as pd, numpy as np
from analysis.scratch_coder_stage_a import validate_export, derive_sufficiency_subsets, summarise_taxonomy_fit
from analysis.scratch_coder_stage_a.api import build_panels, run_replacement_analysis
ROOT = Path.cwd()
OUT = ROOT / 'analysis/outputs_validation_scratch_20260824'
metadata = json.loads((OUT/'run_metadata.json').read_text())
{key: metadata[key] for key in ['raw_export_relative_path','raw_export_sha256','raw_export_rows','raw_export_columns','verified_authorities']}

## 2. Panel QA

In [ ]:
qa = pd.read_csv(OUT/'qa_summary.csv')
qa

## 3. Field mapping

In [ ]:
pd.read_csv(OUT/'field_mapping.csv')

## 4. Agreement implementation

In [ ]:
from analysis.validation.metrics import masi_distance
from analysis.validation.alpha import krippendorff_alpha
examples = {'identical': float(masi_distance(frozenset({'A'}), frozenset({'A'}))), 'partial': float(masi_distance(frozenset({'A'}), frozenset({'A','B'}))), 'disjoint': float(masi_distance(frozenset({'A'}), frozenset({'B'})))}
assert examples['identical'] < examples['partial'] < examples['disjoint']
examples

## 5. Baseline point estimates

In [ ]:
data = validate_export()
subsets = derive_sufficiency_subsets(data)
point_rows=[]
for dimension in ['Research Domains','Analytical Purposes','Demographic disparities / equity','COVID-19 & Pandemic']:
    result=run_replacement_analysis(build_panels(data,data.baseline_ids,dimension),dimension)
    point_rows.append({'dimension':dimension,'N':len(result.common_record_ids),'ABC':result.alpha_abc.alpha,'LBC':result.alpha_lbc.alpha,'ALC':result.alpha_alc.alpha,'ABL':result.alpha_abl.alpha,'delta_min':result.delta_min})
pd.DataFrame(point_rows)

## 6. Bootstrap

In [ ]:
reps=pd.read_csv(OUT/'bootstrap_replicates.csv')
deltas=pd.read_csv(OUT/'replacement_delta_results.csv')
manual=(reps[reps.population.eq('baseline')].groupby('dimension')['delta_min'].quantile([.025,.975],interpolation='linear').unstack())
reported=deltas[(deltas.population.eq('baseline')) & (deltas.delta.eq('delta_min'))].set_index('dimension')[['ci_lower','ci_upper']]
assert np.allclose(manual.sort_index().to_numpy(),reported.sort_index().to_numpy(),equal_nan=True)
manual

## 7. Sufficiency

In [ ]:
from analysis.scratch_coder_stage_a.sufficiency import summarise_sufficiency
suff=summarise_sufficiency(data)
pd.DataFrame(suff['records'])

## 8. Broad/strict subsets

In [ ]:
{'broad_register_usable':len(subsets['broad']),'strict_register_sufficient':len(subsets['strict'])}

## 9. Conditioned replacement

In [ ]:
pd.read_csv(OUT/'replacement_delta_results.csv').query("population in ['baseline_broad_usable','baseline_strict_sufficient'] and delta == 'delta_min'")

## 10. Taxonomy fit

In [ ]:
tax=summarise_taxonomy_fit(data)
pd.DataFrame(tax['records'])

## 11. Timing

In [ ]:
pd.read_csv(OUT/'timing_summary.csv')

## 12. Cross-checks

In [ ]:
saved=pd.read_csv(OUT/'replacement_panel_results.csv')
calc=pd.DataFrame(point_rows).set_index('dimension')
base=saved[saved.population.eq('baseline')].pivot(index='dimension',columns='panel',values='point_estimate')
assert np.allclose(calc.loc[base.index,['ABC','LBC','ALC','ABL']],base[['ABC','LBC','ALC','ABL']])
assert len(data.baseline_ids)==150 and len(data.hard_case_ids)==75 and len(data.responses)==675
'All aggregate cross-checks passed.'